# Observing Agents in AutoGen

Use **`agent.on_messages(...)`** to see what an agent did.

It returns a `Response` with two parts:
- `inner_messages` — tool calls / tool results
- `chat_message` — the final reply

## Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found. Add it to .env"

from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")

def add(a: float, b: float) -> float:
    """Add two numbers and return the sum."""
    return a + b

agent = AssistantAgent(
    name="mathbot",
    model_client=model_client,
    tools=[add],
    system_message="Always use the add tool for arithmetic.",
)

print("Ready.")

## Run with `on_messages`

In [ ]:
from autogen_core import CancellationToken
from autogen_agentchat.messages import TextMessage

response = await agent.on_messages(
    [TextMessage(content="What is 17 + 25?", source="user")],
    cancellation_token=CancellationToken(),
)

for m in response.inner_messages or []:
    print(f"[{m.type}] {m.content}")

print(f"\nFinal: {response.chat_message.content}")

## Live view with `Console`

`on_messages_stream(...)` yields events as they happen. Pipe it into `Console` for a nicely formatted live view.

In [ ]:
from autogen_agentchat.ui import Console

await Console(agent.on_messages_stream(
    [TextMessage(content="What is 100 + 250?", source="user")],
    cancellation_token=CancellationToken(),
))

## Monitor token usage

Each LLM event has a `models_usage` field with `prompt_tokens` and `completion_tokens`. The model client also tracks cumulative usage across every call.

In [ ]:
response = await Console(agent.on_messages_stream(
    [TextMessage(content="What is 42 + 58?", source="user")],
    cancellation_token=CancellationToken(),
))

prompt = completion = 0
for m in list(response.inner_messages or []) + [response.chat_message]:
    u = getattr(m, "models_usage", None)
    if u is not None:
        print(f"[{m.type}] prompt={u.prompt_tokens}  completion={u.completion_tokens}")
        prompt += u.prompt_tokens
        completion += u.completion_tokens

print(f"\nThis run:    prompt={prompt}  completion={completion}")

total = model_client.total_usage()
print(f"Cumulative:  prompt={total.prompt_tokens}  completion={total.completion_tokens}")